# Notebook 03 — RoBERTa Sentiment Classification

**Project**: NLP Business Case — Automated Customer Reviews  
**Bootcamp**: Ironhack  
**Model**: `roberta-base` (FacebookAI)  
**Task**: Fine-tune RoBERTa for 3-class sentiment classification (Negative / Neutral / Positive)  
**Purpose**: High-performance comparison model versus the DistilBERT baseline built in Notebook 02

---

## Why RoBERTa?

In this notebook, I fine-tune **RoBERTa** (`roberta-base`) — a robustly optimized version of BERT developed by Facebook AI. Compared to the DistilBERT model trained in Notebook 02, RoBERTa offers:

- **~125M parameters** (vs ~66M for DistilBERT) — deeper representational capacity
- **12 transformer layers** trained on **160GB+ of text data** (BooksCorpus, CC-News, OpenWebText, Stories)
- **No Next Sentence Prediction (NSP)** task — removes a training objective BERT authors later showed was not useful
- **Larger batches + dynamic masking** — more robust pretraining signal
- Consistently **outperforms BERT on GLUE, SQuAD, and RACE** benchmarks

The cost is speed and memory: RoBERTa is ~2× the size of DistilBERT, requires smaller batch sizes, and trains more slowly. This notebook quantifies whether that cost is worth it.

---

## Notebook Sections

| # | Section | Description |
|---|---------|-------------|
| 0 | Setup & Environment | Drive mount, installs, imports, paths, seeds |
| 1 | Load Dataset | Load Arrow dataset from NB01, verify splits, class weights |
| 2 | Tokenization | RobertaTokenizerFast, BPE encoding, tokenize all splits |
| 3 | Model Setup | Load RobertaForSequenceClassification, GPU check, param count |
| 4 | Training | WeightedTrainer, TrainingArguments, EarlyStopping |
| 5 | Evaluation | Test metrics, confusion matrix, classification report |
| 6 | Model Comparison | RoBERTa vs DistilBERT side-by-side analysis |
| 7 | Inference & Save | Predictions CSV, save fine-tuned model |
| 8 | Results Summary | Final metrics, sample predictions, loss curves |

---
## Section 0 — Setup & Environment

In this section, I mount Google Drive (when running on Colab), install all required libraries, import every dependency used throughout the notebook, define all directory paths, and set random seeds to ensure reproducibility across runs.

In [ ]:
# ---------------------------------------------------------------------------
# 0.1  Detect execution environment (Colab vs local)
# ---------------------------------------------------------------------------

# Try to import the google.colab module — only available inside a Colab runtime
try:
    import google.colab          # noqa: F401  (imported for side-effect check only)
    IN_COLAB = True              # Flag: we are running inside Google Colab
except ImportError:
    IN_COLAB = False             # Flag: we are running locally

print(f"Running in Colab: {IN_COLAB}")  # Expected output: True (Colab) or False (local)

In [ ]:
# ---------------------------------------------------------------------------
# 0.2  Mount Google Drive (Colab only)
# ---------------------------------------------------------------------------
# Mounting Drive makes all Drive files accessible under /content/drive/MyDrive
# This prevents data loss when the Colab runtime disconnects or resets

if IN_COLAB:
    from google.colab import drive          # Import the Colab Drive utility
    drive.mount('/content/drive')           # Mount Drive at /content/drive
    # Expected output: "Mounted at /content/drive" (after OAuth approval)

In [ ]:
# ---------------------------------------------------------------------------
# 0.3  Install required Python packages
# ---------------------------------------------------------------------------
# Install all libraries needed for this notebook in one pip call
# --quiet suppresses verbose output; -U ensures latest compatible versions

if IN_COLAB:
    # Install the full NLP/ML stack required for fine-tuning and evaluation
    # transformers  : HuggingFace Transformers — RoBERTa model + tokenizer + Trainer
    # datasets      : HuggingFace Datasets — Arrow format loading
    # torch         : PyTorch — deep learning backend
    # scikit-learn  : metrics (accuracy, F1, confusion matrix, classification report)
    # matplotlib    : plotting (confusion matrix, loss curves, comparison bar chart)
    # seaborn       : styled heatmaps for confusion matrix visualization
    # tqdm          : progress bars for inference loops
    # accelerate    : HuggingFace Accelerate — required by Trainer for mixed precision
    import subprocess
    subprocess.run(
        [
            "pip", "install", "-q", "-U",
            "transformers",
            "datasets",
            "torch",
            "scikit-learn",
            "matplotlib",
            "seaborn",
            "tqdm",
            "accelerate",
        ],
        check=True,  # Raise CalledProcessError if pip fails
    )
    print("All packages installed successfully.")  # Expected: confirmation message

In [ ]:
# ---------------------------------------------------------------------------
# 0.4  Standard library and third-party imports
# ---------------------------------------------------------------------------

# --- Standard library ---
import os           # File system operations (path creation, existence checks)
import json         # Save/load metrics as JSON files
import random       # Random seed for Python's built-in RNG
import warnings     # Suppress non-critical warnings from dependencies

# --- Numerical computing ---
import numpy as np  # Array operations, random seeding, argmax for predictions

# --- Data manipulation ---
import pandas as pd  # DataFrames for predictions CSV and comparison tables

# --- Visualization ---
import matplotlib.pyplot as plt  # Base plotting engine for all figures
import seaborn as sns            # Heatmap styling for the confusion matrix

# --- HuggingFace Datasets ---
from datasets import DatasetDict  # Load Arrow-format dataset saved by NB01

# --- HuggingFace Transformers — Tokenizer ---
from transformers import RobertaTokenizerFast  # Fast BPE tokenizer for roberta-base

# --- HuggingFace Transformers — Model ---
from transformers import RobertaForSequenceClassification  # RoBERTa with classification head

# --- HuggingFace Transformers — Training infrastructure ---
from transformers import (
    Trainer,               # High-level training loop
    TrainingArguments,     # Hyperparameter configuration for Trainer
    DataCollatorWithPadding,  # Dynamic padding collator — more efficient than static padding
    EarlyStoppingCallback,    # Stop training when eval metric stops improving
)

# --- PyTorch ---
import torch                         # Core tensor operations, device management
import torch.nn as nn                # Neural network module — used for CrossEntropyLoss
import torch.nn.functional as F      # Functional API — used for softmax in inference

# --- scikit-learn metrics ---
from sklearn.metrics import (
    accuracy_score,          # Fraction of correct predictions
    precision_recall_fscore_support,  # Per-class P/R/F1 with weighting options
    confusion_matrix,        # NxN matrix of true vs predicted labels
    classification_report,   # Full per-class report as formatted string
    f1_score,                # Weighted F1 for compute_metrics callback
)
from sklearn.utils.class_weight import compute_class_weight  # Handles class imbalance

# --- Progress bars ---
from tqdm.auto import tqdm  # Auto-selects tqdm.notebook in Jupyter, tqdm.tqdm elsewhere

# Suppress FutureWarnings from transformers internals — keeps output clean
warnings.filterwarnings("ignore", category=FutureWarning)

print("All imports successful.")  # Expected: confirmation line

In [ ]:
# ---------------------------------------------------------------------------
# 0.5  Directory path definitions
# ---------------------------------------------------------------------------
# All artifacts (datasets, models, plots, CSVs, JSONs) are stored on Google Drive
# so they persist across Colab session resets.

if IN_COLAB:
    # Root of the project inside Google Drive
    BASE_DIR = "/content/drive/MyDrive/nlp-project/business-case-01"  # Update path if folder differs
else:
    # When running locally, use the repository root (one level above this notebook)
    BASE_DIR = os.path.dirname(os.path.abspath("__file__"))  # Resolves to repo root

# Directory where NB01 saved the preprocessed Arrow dataset
OUTPUT_DIR = os.path.join(BASE_DIR, "output")  # e.g. .../nlp-project/business-case-01/output

# Directory for all plot images (PNG files)
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")  # e.g. .../output/plots

# Directory for fine-tuned model weights and tokenizer
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")  # e.g. .../output/models

# Path where the fine-tuned RoBERTa model will be saved after training
ROBERTA_MODEL_DIR = os.path.join(MODELS_DIR, "roberta_sentiment")  # model subfolder

# ---------------------------------------------------------------------------
# Create all directories if they don't already exist
# ---------------------------------------------------------------------------
for directory in [OUTPUT_DIR, PLOTS_DIR, MODELS_DIR, ROBERTA_MODEL_DIR]:
    os.makedirs(directory, exist_ok=True)  # exist_ok=True prevents error if dir exists

# Print all resolved paths so I can verify they look correct before continuing
print(f"BASE_DIR   : {BASE_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"PLOTS_DIR  : {PLOTS_DIR}")
print(f"MODELS_DIR : {MODELS_DIR}")
print(f"ROBERTA_DIR: {ROBERTA_MODEL_DIR}")
# Expected: All 5 paths printed cleanly; directories created if missing

In [ ]:
# ---------------------------------------------------------------------------
# 0.6  Set random seeds for full reproducibility
# ---------------------------------------------------------------------------
# Machine learning experiments are stochastic by default (weight initialization,
# dropout, data shuffling). Fixing seeds ensures the same results every run.

SEED = 42  # Canonical seed used throughout all notebooks in this project

random.seed(SEED)          # Seed Python's built-in random module
np.random.seed(SEED)       # Seed NumPy's RNG (used by scikit-learn internally)
torch.manual_seed(SEED)    # Seed PyTorch CPU operations

# Seed GPU operations if CUDA is available
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)    # Seeds all GPU devices
    # Force deterministic CUDA algorithms (slight performance cost, worth it for reproducibility)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # Disables auto-tuner that introduces randomness

print(f"Random seeds fixed to {SEED} for Python, NumPy, and PyTorch.")
# Expected: confirmation message; no errors

---
## Section 1 — Load Dataset

In this section, I load the preprocessed Arrow-format dataset produced by Notebook 01. Using HuggingFace's `DatasetDict.load_from_disk()` reads directly from the binary Arrow files — much faster than re-reading a CSV and far more memory-efficient at scale. I then verify the splits, inspect class distributions, review sample records, and compute class weights to handle any label imbalance during training.

In [ ]:
# ---------------------------------------------------------------------------
# 1.1  Load the Arrow-format dataset saved by Notebook 01
# ---------------------------------------------------------------------------

# Path to the directory where NB01 saved the DatasetDict via .save_to_disk()
DATASET_PATH = os.path.join(OUTPUT_DIR, "dataset")  # e.g. .../output/dataset

# Load the full DatasetDict (train / validation / test splits) from disk
# DatasetDict.load_from_disk() reads Arrow binary files — no re-parsing needed
dataset = DatasetDict.load_from_disk(DATASET_PATH)

print("Dataset loaded successfully.")
print(dataset)  # Expected: DatasetDict with train/validation/test keys + feature schema

In [ ]:
# ---------------------------------------------------------------------------
# 1.2  Verify split sizes and feature schema
# ---------------------------------------------------------------------------

# Iterate over each split and print its size and feature names
for split_name, split_data in dataset.items():
    print(f"{split_name:12s} — {len(split_data):>8,} rows | features: {list(split_data.features.keys())}")
    # Expected: train (~80%), validation (~10%), test (~10%) of total dataset

In [ ]:
# ---------------------------------------------------------------------------
# 1.3  Label mapping — same convention as NB01 and NB02
# ---------------------------------------------------------------------------
# Sentiment labels were assigned in NB01 based on star rating:
#   1-2 stars → 0 (Negative)
#   3 stars   → 1 (Neutral)
#   4-5 stars → 2 (Positive)

LABEL2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}  # String label → integer ID
ID2LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}   # Integer ID → string label
NUM_LABELS = len(LABEL2ID)                                  # 3 classes total

print(f"Label mapping: {LABEL2ID}")
print(f"Number of classes: {NUM_LABELS}")
# Expected: {'Negative': 0, 'Neutral': 1, 'Positive': 2} and 3

In [ ]:
# ---------------------------------------------------------------------------
# 1.4  Inspect class distribution across splits
# ---------------------------------------------------------------------------
# Understanding class imbalance is critical — if one class dominates (e.g., Positive
# reviews vastly outnumber Negative), the model may learn to predict the majority
# class most of the time. I will use class weights in Section 4 to counteract this.

for split_name, split_data in dataset.items():
    # Convert the label column of this split to a NumPy array for counting
    labels_array = np.array(split_data["label"])  # Shape: (n_samples,)

    # Count occurrences of each label (bincount works for non-negative integers)
    counts = np.bincount(labels_array, minlength=NUM_LABELS)  # Shape: (3,)

    # Compute percentage of each class for easier interpretation
    percentages = counts / counts.sum() * 100  # Normalize to [0, 100]

    print(f"\n[{split_name.upper()}] Class distribution:")
    for label_id, count in enumerate(counts):
        label_name = ID2LABEL[label_id]  # Convert int → string name
        print(f"  {label_name:10s} (id={label_id}): {count:>8,}  ({percentages[label_id]:.1f}%)")
# Expected: class distribution consistent across splits; Positive likely dominant

In [ ]:
# ---------------------------------------------------------------------------
# 1.5  Show sample records from the training split
# ---------------------------------------------------------------------------
# Reviewing raw samples confirms the text field is present and labels look correct

print("Sample records from the training split:\n")
for idx in range(5):  # Show the first 5 rows
    sample = dataset["train"][idx]       # Dict with 'text', 'label', possibly other fields
    label_name = ID2LABEL[sample["label"]]  # Convert integer label to readable string
    # Truncate text to 120 chars for display — full text may be hundreds of tokens
    text_preview = sample["text"][:120].replace("\n", " ")
    print(f"[{idx}] Label={label_name:8s} | Text: {text_preview}...")
# Expected: 5 rows each showing a label (Negative/Neutral/Positive) and review text snippet

In [ ]:
# ---------------------------------------------------------------------------
# 1.6  Compute class weights to handle class imbalance
# ---------------------------------------------------------------------------
# If Positive reviews are 5× more common than Negative, the model is incentivized
# to always predict Positive. Class weighting penalizes errors on minority classes
# more heavily, forcing the model to pay attention to them during training.
#
# sklearn's compute_class_weight uses the formula:
#   weight_c = n_samples / (n_classes * count_c)
# So minority classes get higher weights.

# Extract all training labels as a NumPy array
train_labels = np.array(dataset["train"]["label"])  # Shape: (n_train,)

# Compute balanced class weights using sklearn
class_weights_array = compute_class_weight(
    class_weight="balanced",           # Uses the balanced formula above
    classes=np.arange(NUM_LABELS),     # [0, 1, 2] — all class IDs
    y=train_labels,                    # Array of training labels
)  # Returns a NumPy array of shape (3,) with one weight per class

# Convert to a PyTorch float tensor so it can be passed to CrossEntropyLoss
class_weights_tensor = torch.tensor(class_weights_array, dtype=torch.float32)

print("Class weights (higher = rarer class, penalized more):")
for label_id, weight in enumerate(class_weights_array):
    print(f"  {ID2LABEL[label_id]:10s} (id={label_id}): weight = {weight:.4f}")
# Expected: Neutral likely has the highest weight (rarest), Positive the lowest (most common)

---
## Section 2 — Tokenization

In this section, I load the `RobertaTokenizerFast` and apply it to all splits. 

**Key difference from DistilBERT**: RoBERTa uses **Byte Pair Encoding (BPE)** tokenization, while DistilBERT uses **WordPiece**. BPE builds its vocabulary by iteratively merging the most frequent byte/character pairs, whereas WordPiece maximizes the likelihood of the training data. In practice:
- BPE operates at the byte level — it handles any Unicode character without an `[UNK]` token
- RoBERTa uses `<s>` and `</s>` as special tokens (vs BERT's `[CLS]` and `[SEP]`)
- RoBERTa adds a space prefix to subwords when they appear mid-sentence (the `Ġ` prefix)

I set `max_length=128` for consistency with NB02, enabling a direct performance comparison.

In [ ]:
# ---------------------------------------------------------------------------
# 2.1  Load the RoBERTa fast tokenizer
# ---------------------------------------------------------------------------
# RobertaTokenizerFast is the Rust-backed tokenizer — significantly faster than
# the pure-Python RobertaTokenizer, especially when processing large datasets

TOKENIZER_NAME = "roberta-base"  # HuggingFace model hub identifier

tokenizer = RobertaTokenizerFast.from_pretrained(
    TOKENIZER_NAME,     # Load vocabulary and tokenizer config from HuggingFace hub
    add_prefix_space=False,  # Don't add a leading space before all inputs
)

print(f"Tokenizer loaded: {TOKENIZER_NAME}")
print(f"Vocabulary size : {tokenizer.vocab_size:,}")  # Expected: 50,265 tokens
print(f"Special tokens  : {tokenizer.all_special_tokens}")  # Expected: ['<s>', '</s>', '<unk>', '<pad>', '<mask>']
print(f"Max model length: {tokenizer.model_max_length}")

In [ ]:
# ---------------------------------------------------------------------------
# 2.2  Define the tokenization function
# ---------------------------------------------------------------------------
# This function will be applied to every row in the dataset via .map()
# max_length=128 keeps memory usage manageable for a 125M-parameter model
# truncation=True cuts sequences longer than max_length (common for long reviews)
# padding='max_length' pads shorter sequences to exactly max_length
#   NOTE: DataCollatorWithPadding in Section 4 will override per-batch padding,
#   which is more efficient. We use max_length padding here so Arrow caches the
#   tokenized dataset with fixed-size tensors.

MAX_LENGTH = 128  # Maximum token sequence length (same as NB02 for fair comparison)

def tokenize_function(examples):
    """Tokenize a batch of text examples using the RoBERTa BPE tokenizer.

    Args:
        examples (dict): Batch dict with key 'text' containing a list of strings.

    Returns:
        dict: Tokenizer output with 'input_ids', 'attention_mask' fields.
    """
    return tokenizer(
        examples["text"],          # Text column from the dataset batch
        truncation=True,           # Cut sequences longer than max_length tokens
        padding="max_length",      # Pad all sequences to exactly max_length
        max_length=MAX_LENGTH,     # Maximum token count (128 tokens per review)
        return_tensors=None,       # Return Python lists, not tensors (Arrow-compatible)
    )  # Returns: {'input_ids': [...], 'attention_mask': [...]}

In [ ]:
# ---------------------------------------------------------------------------
# 2.3  Apply tokenizer to all splits using batched mapping
# ---------------------------------------------------------------------------
# .map() applies tokenize_function to the entire dataset in batches
# batched=True passes multiple rows at once (faster than row-by-row)
# remove_columns=["text"] drops the raw text column after tokenization
#   to reduce memory — the model only needs input_ids and attention_mask

# Identify which column(s) to remove after tokenization
# We keep 'label' since the Trainer needs it for loss computation
columns_to_remove = [col for col in dataset["train"].column_names if col != "label"]

print(f"Columns before tokenization: {dataset['train'].column_names}")
print(f"Columns to remove after tokenization: {columns_to_remove}")

# Apply tokenization across all splits simultaneously
tokenized_dataset = dataset.map(
    tokenize_function,            # The tokenization function defined above
    batched=True,                 # Process rows in batches for speed
    batch_size=1000,              # Process 1000 rows per batch
    remove_columns=columns_to_remove,  # Drop raw text to save memory
    desc="Tokenizing dataset",    # Progress bar label
)  # Expected: DatasetDict with input_ids, attention_mask, label columns

print("\nTokenization complete.")
print(f"Columns after tokenization: {tokenized_dataset['train'].column_names}")
# Expected: ['input_ids', 'attention_mask', 'label'] (token_type_ids absent — RoBERTa doesn't use them)

In [ ]:
# ---------------------------------------------------------------------------
# 2.4  Set dataset format to PyTorch tensors
# ---------------------------------------------------------------------------
# By default HuggingFace datasets store data as Python lists
# Setting format to 'torch' makes __getitem__ return tensors directly
# This is required by the Trainer and DataCollatorWithPadding

tokenized_dataset.set_format(
    type="torch",                                         # Return torch.Tensor objects
    columns=["input_ids", "attention_mask", "label"],    # Only these columns as tensors
)  # Expected: no output; subsequent indexing returns tensors

print("Dataset format set to PyTorch tensors.")
# Verify by checking the type of one element
sample_item = tokenized_dataset["train"][0]
print(f"input_ids type    : {type(sample_item['input_ids'])}")
print(f"input_ids shape   : {sample_item['input_ids'].shape}")
print(f"attention_mask    : {sample_item['attention_mask'].shape}")
print(f"label             : {sample_item['label']} → {ID2LABEL[sample_item['label'].item()]}")
# Expected: input_ids shape = torch.Size([128]); label is a scalar tensor

In [ ]:
# ---------------------------------------------------------------------------
# 2.5  Inspect a tokenized example and compare BPE vs WordPiece
# ---------------------------------------------------------------------------
# Decoding the input_ids back to tokens lets me verify the BPE tokenization
# and note differences from DistilBERT's WordPiece tokenizer used in NB02

# Grab the first training sample's raw text (from the original un-tokenized dataset)
raw_text = dataset["train"][0]["text"]

# Tokenize the sample and retrieve the token list
encoded = tokenizer(
    raw_text,
    truncation=True,
    max_length=MAX_LENGTH,
    padding="max_length",
)

# Convert input_ids back to readable tokens
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][:20])  # First 20 tokens

print(f"Original text (first 120 chars): {raw_text[:120]}")
print(f"\nFirst 20 BPE tokens: {tokens}")
print("""
RoBERTa (BPE) vs DistilBERT (WordPiece) tokenization differences:
  - BPE uses <s> / </s> as CLS/SEP special tokens; WordPiece uses [CLS] / [SEP]
  - BPE subwords that follow a space are prefixed with 'Ġ' (e.g., 'Ġgreat')
  - WordPiece continuation tokens use '##' prefix (e.g., '##ing')
  - BPE operates at the byte level — no <unk> for unknown characters
  - RoBERTa vocabulary: 50,265 tokens vs DistilBERT: 30,522 tokens
""")
# Expected: token list showing <s>, Ġ-prefixed words, and </s> or <pad> at end

---
## Section 3 — Model Setup

In this section, I load `RobertaForSequenceClassification` with a 3-class classification head, attach the label mappings, move the model to GPU if available, and print a parameter count comparison against the DistilBERT baseline.

In [ ]:
# ---------------------------------------------------------------------------
# 3.1  Detect and configure compute device (GPU vs CPU)
# ---------------------------------------------------------------------------

# Check whether a CUDA-capable GPU is available in this runtime
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Compute device: {DEVICE}")

if DEVICE.type == "cuda":
    # Print GPU model name and available VRAM for reference
    gpu_name = torch.cuda.get_device_name(0)     # Name of the first GPU (e.g., T4, V100)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9  # Convert bytes to GB
    print(f"GPU model : {gpu_name}")
    print(f"GPU memory: {total_mem:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")
    print("Enable GPU in Colab: Runtime → Change runtime type → GPU")
# Expected: CUDA device with 12–16 GB VRAM (T4 or L4 in Colab)

In [ ]:
# ---------------------------------------------------------------------------
# 3.2  Load RobertaForSequenceClassification
# ---------------------------------------------------------------------------
# RobertaForSequenceClassification = roberta-base encoder + a linear classification head
# The classification head consists of:
#   - A dense layer: hidden_size (768) → hidden_size (768) with tanh activation
#   - A dropout layer
#   - A projection layer: hidden_size (768) → num_labels (3)
# This replaces the Masked Language Modeling head from pretraining

MODEL_NAME = "roberta-base"  # FacebookAI's RoBERTa base model identifier

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,           # Download pretrained weights from HuggingFace hub
    num_labels=NUM_LABELS,  # 3 output neurons (Negative / Neutral / Positive)
    id2label=ID2LABEL,    # Attach ID→label mapping to model config
    label2id=LABEL2ID,    # Attach label→ID mapping to model config
)  # Expected: warning about "Some weights... newly initialized" — normal for classification head

# Move model to the available compute device
model = model.to(DEVICE)  # Transfers all 125M parameter tensors to GPU memory

print(f"Model loaded and moved to {DEVICE}.")

In [ ]:
# ---------------------------------------------------------------------------
# 3.3  Count model parameters and compare with DistilBERT
# ---------------------------------------------------------------------------
# Understanding parameter count helps explain training time and memory requirements

# Count total trainable parameters (those with requires_grad=True)
total_params = sum(p.numel() for p in model.parameters())          # All parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)  # Trainable only

print("=" * 55)
print(f"  Model               : {MODEL_NAME}")
print(f"  Total parameters    : {total_params:>15,}")
print(f"  Trainable parameters: {trainable_params:>15,}")
print("=" * 55)
print(f"  DistilBERT baseline : ~66,000,000  (NB02)")
print(f"  RoBERTa (this model): ~{total_params:,}")
print(f"  RoBERTa is ~{total_params / 66_000_000:.1f}× larger than DistilBERT")
print("=" * 55)
print("""
Architecture highlights:
  - 12 transformer encoder layers (vs 6 in DistilBERT)
  - 768 hidden dimensions per layer
  - 12 self-attention heads per layer
  - Feed-forward dimension: 3,072
  - Trained on 160GB text (BooksCorpus, CC-News, OpenWebText, Stories)
  - No Next Sentence Prediction task (NSP removed vs original BERT)
""")
# Expected: ~125M total parameters; ~1.9× larger than DistilBERT

---
## Section 4 — Training

In this section, I configure and run the fine-tuning process. I use a custom `WeightedTrainer` that replaces the standard CrossEntropyLoss with a class-weighted version to address label imbalance. The training configuration uses a lower learning rate (1e-5) than DistilBERT (2e-5) because RoBERTa is more sensitive — its larger weights are already well-calibrated from pretraining, so aggressive updates can destroy useful representations.

In [ ]:
# ---------------------------------------------------------------------------
# 4.1  Define the compute_metrics function for the Trainer callback
# ---------------------------------------------------------------------------
# The Trainer calls compute_metrics after each evaluation step
# It receives an EvalPrediction object with .predictions and .label_ids

def compute_metrics(eval_pred):
    """Compute accuracy and weighted F1 score from Trainer evaluation predictions.

    Args:
        eval_pred: EvalPrediction namedtuple with fields:
            - predictions: np.ndarray of shape (n_samples, n_classes) — raw logits
            - label_ids: np.ndarray of shape (n_samples,) — true integer labels

    Returns:
        dict: {'accuracy': float, 'f1': float} — both in [0, 1]
    """
    logits, labels = eval_pred.predictions, eval_pred.label_ids

    # Convert logits to predicted class IDs via argmax along class dimension
    predictions = np.argmax(logits, axis=-1)  # Shape: (n_samples,) — integer class predictions

    # Compute fraction of exactly correct predictions
    acc = accuracy_score(labels, predictions)  # Float in [0, 1]

    # Compute weighted F1: weights each class by its support (# true samples)
    # 'weighted' handles class imbalance better than 'macro' for reporting
    f1 = f1_score(labels, predictions, average="weighted")  # Float in [0, 1]

    return {"accuracy": acc, "f1": f1}  # Trainer logs these after each eval epoch

In [ ]:
# ---------------------------------------------------------------------------
# 4.2  Define the WeightedTrainer with class-weighted CrossEntropyLoss
# ---------------------------------------------------------------------------
# The default HuggingFace Trainer uses unweighted CrossEntropyLoss
# I override compute_loss() to inject class weights, penalizing errors
# on minority classes (Negative, Neutral) more heavily than Positive

class WeightedTrainer(Trainer):
    """Custom Trainer subclass that uses class-weighted CrossEntropyLoss.

    This overrides the standard Trainer.compute_loss() method to inject
    class_weights_tensor into torch.nn.CrossEntropyLoss, which upweights
    rare classes (Negative, Neutral) and downweights common ones (Positive).
    """

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Compute weighted cross-entropy loss for a training batch.

        Args:
            model: The RoBERTa model being fine-tuned.
            inputs: Dict of tensors from the DataCollator (input_ids, attention_mask, labels).
            return_outputs: If True, also return model outputs (used during evaluation).

        Returns:
            loss (torch.Tensor scalar) or (loss, outputs) tuple.
        """
        # Extract integer labels from the batch inputs
        labels = inputs.pop("labels")  # Shape: (batch_size,) — integer class IDs

        # Forward pass: feed input_ids and attention_mask to the model
        outputs = model(**inputs)  # Returns ModelOutput with .logits attribute

        # Extract raw logits from model output (pre-softmax class scores)
        logits = outputs.logits  # Shape: (batch_size, num_labels=3)

        # Instantiate CrossEntropyLoss with class weights on the correct device
        # weight tensor must be on same device as logits
        loss_fn = nn.CrossEntropyLoss(
            weight=class_weights_tensor.to(logits.device)  # Move weights to GPU
        )

        # Compute scalar loss: average weighted cross-entropy over batch
        loss = loss_fn(logits, labels)  # Scalar tensor

        # Return loss alone or (loss, outputs) depending on flag
        return (loss, outputs) if return_outputs else loss

In [ ]:
# ---------------------------------------------------------------------------
# 4.3  Define training hyperparameters via TrainingArguments
# ---------------------------------------------------------------------------
# These values are tuned for RoBERTa specifically:
#   - batch_size=16: smaller than DistilBERT (32) due to 2× model size
#   - learning_rate=1e-5: lower than DistilBERT (2e-5) — RoBERTa is more sensitive
#   - epochs=3: standard for fine-tuning; EarlyStopping will stop earlier if needed

# Directory where checkpoints are saved during training (within OUTPUT_DIR for persistence)
CHECKPOINTS_DIR = os.path.join(OUTPUT_DIR, "roberta_checkpoints")  # Checkpoint save path

BATCH_SIZE = 16         # Per-device batch size — smaller than DistilBERT due to model size
NUM_EPOCHS = 3          # Maximum number of full passes over the training data
LEARNING_RATE = 1e-5    # Fine-tuning LR for RoBERTa (lower than DistilBERT's 2e-5)
WEIGHT_DECAY = 0.01     # L2 regularization coefficient — prevents overfitting

# Compute total training steps (needed to set warmup_steps)
# total_steps = ceil(train_size / batch_size) * num_epochs
steps_per_epoch = len(tokenized_dataset["train"]) // BATCH_SIZE  # Floor division
total_training_steps = steps_per_epoch * NUM_EPOCHS              # All epochs
warmup_steps = int(0.10 * total_training_steps)                  # 10% warmup

print(f"Training samples   : {len(tokenized_dataset['train']):,}")
print(f"Steps per epoch    : {steps_per_epoch:,}")
print(f"Total train steps  : {total_training_steps:,}")
print(f"Warmup steps (10%) : {warmup_steps:,}")

training_args = TrainingArguments(
    output_dir=CHECKPOINTS_DIR,          # Save checkpoints and logs here
    num_train_epochs=NUM_EPOCHS,         # Train for up to 3 epochs
    per_device_train_batch_size=BATCH_SIZE,   # 16 samples per GPU step
    per_device_eval_batch_size=BATCH_SIZE * 2,  # 32 for eval — no gradients, fits 2× batch
    learning_rate=LEARNING_RATE,         # AdamW base learning rate (1e-5 for RoBERTa)
    weight_decay=WEIGHT_DECAY,           # L2 regularization on all non-bias parameters
    warmup_steps=warmup_steps,           # Linear warmup over first 10% of steps
    eval_strategy="epoch",              # Run evaluation at end of every epoch
    save_strategy="epoch",              # Save checkpoint at end of every epoch
    load_best_model_at_end=True,        # Restore best checkpoint when training ends
    metric_for_best_model="eval_loss",  # Use validation loss to select best model
    greater_is_better=False,            # Lower eval_loss = better model
    logging_dir=os.path.join(CHECKPOINTS_DIR, "logs"),  # TensorBoard log directory
    logging_steps=50,                   # Log training loss every 50 steps
    seed=SEED,                          # Reproducibility seed for Trainer internals
    fp16=torch.cuda.is_available(),     # Mixed-precision (FP16) training — faster on GPU
    dataloader_num_workers=2,           # Background workers for data loading
    report_to="none",                   # Disable WandB/TensorBoard external reporting
    save_total_limit=2,                 # Keep only last 2 checkpoints to save disk space
)

print("\nTrainingArguments configured successfully.")
# Expected: confirmation; all hyperparameters set

In [ ]:
# ---------------------------------------------------------------------------
# 4.4  Initialize the DataCollatorWithPadding
# ---------------------------------------------------------------------------
# DataCollatorWithPadding dynamically pads each batch to the length of its
# longest sequence, rather than always padding to max_length=128
# This is more efficient: shorter batches use less memory and compute
# Note: since we already padded to max_length in tokenize_function, the
# collator sees uniform-length sequences — no further padding is applied

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,    # Uses the same tokenizer for pad token ID reference
    padding=True,           # Dynamically pad each batch to its max sequence length
)  # Expected: collator object ready for use by Trainer

print("DataCollatorWithPadding initialized.")

In [ ]:
# ---------------------------------------------------------------------------
# 4.5  Instantiate the WeightedTrainer
# ---------------------------------------------------------------------------

trainer = WeightedTrainer(
    model=model,                                    # RoBERTa classification model
    args=training_args,                             # All hyperparameters defined above
    train_dataset=tokenized_dataset["train"],       # Training split (Arrow format)
    eval_dataset=tokenized_dataset["validation"],   # Validation split for periodic eval
    tokenizer=tokenizer,                            # Needed by Trainer for tokenizer config
    data_collator=data_collator,                    # Dynamic padding collator
    compute_metrics=compute_metrics,                # Accuracy + weighted F1 per eval step
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,  # Stop if eval_loss doesn't improve for 2 epochs
            early_stopping_threshold=0.001,  # Minimum improvement to count as progress
        )
    ],
)  # Expected: WeightedTrainer object with all components wired together

print("WeightedTrainer instantiated with EarlyStoppingCallback (patience=2).")

In [ ]:
# ---------------------------------------------------------------------------
# 4.6  Run training
# ---------------------------------------------------------------------------
# This launches the fine-tuning loop. Each epoch:
#   1. Forward pass through RoBERTa with input_ids + attention_mask
#   2. Compute weighted CrossEntropyLoss (custom WeightedTrainer)
#   3. Backward pass — compute gradients via autograd
#   4. AdamW optimizer step — update all 125M parameters
#   5. Learning rate scheduler step (linear warmup then decay)
# After each epoch: evaluate on validation set, save checkpoint

print("Starting RoBERTa fine-tuning...")
print(f"  Epochs       : {NUM_EPOCHS}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Device       : {DEVICE}")

# Launch the training loop
train_result = trainer.train()  # Returns TrainOutput with metrics and state

# Log training metrics
trainer.log_metrics("train", train_result.metrics)  # Print training time and steps
trainer.save_metrics("train", train_result.metrics)  # Save to output_dir/train_results.json

print("\nTraining complete.")
print(f"  Total training time: {train_result.metrics.get('train_runtime', 0):.1f} seconds")
print(f"  Samples per second : {train_result.metrics.get('train_samples_per_second', 0):.1f}")
# Expected: ~30-60 min on a Colab T4 GPU for 3 epochs on a large dataset

---
## Section 5 — Evaluation

In this section, I evaluate the fine-tuned RoBERTa model on the held-out test set. The test set was never seen during training or validation, so it gives an unbiased estimate of real-world performance. I compute all key classification metrics, generate a confusion matrix heatmap, and save everything for the comparison in Section 6.

In [ ]:
# ---------------------------------------------------------------------------
# 5.1  Generate predictions on the test set
# ---------------------------------------------------------------------------
# trainer.predict() runs inference on the entire test set in batches
# It returns raw logits (not probabilities) and the ground-truth labels

print("Running inference on test set...")
test_output = trainer.predict(tokenized_dataset["test"])  # Returns PredictionOutput

# Extract raw logits: shape (n_test, num_labels=3)
test_logits = test_output.predictions  # NumPy array — one row per test sample

# Extract ground-truth labels: shape (n_test,)
test_true_labels = test_output.label_ids  # NumPy array — integer class IDs

# Convert logits to predicted class IDs via argmax
test_pred_labels = np.argmax(test_logits, axis=-1)  # Shape: (n_test,) — integer predictions

# Compute confidence scores: apply softmax to convert logits → probabilities
test_probs = F.softmax(torch.tensor(test_logits, dtype=torch.float32), dim=-1).numpy()
# test_probs shape: (n_test, 3) — probability for each class

# Confidence = max probability across the 3 classes (the model's certainty)
test_confidence = test_probs.max(axis=1)  # Shape: (n_test,) — scalar confidence per sample

print(f"Test samples evaluated: {len(test_true_labels):,}")
# Expected: matches test split size from Section 1

In [ ]:
# ---------------------------------------------------------------------------
# 5.2  Compute all classification metrics
# ---------------------------------------------------------------------------

# --- Accuracy ---
# Fraction of test samples where predicted label == true label
test_accuracy = accuracy_score(test_true_labels, test_pred_labels)  # Float [0, 1]

# --- Precision, Recall, F1 (per class + weighted average) ---
precision, recall, f1, support = precision_recall_fscore_support(
    test_true_labels,
    test_pred_labels,
    average=None,           # Compute per-class metrics (one value per class)
    labels=[0, 1, 2],       # Ensure all 3 classes are included even if not predicted
)  # Returns arrays of shape (3,) for precision, recall, f1; (3,) for support

# --- Weighted averages ---
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    test_true_labels,
    test_pred_labels,
    average="weighted",     # Weighted by support (# true samples per class)
)  # Returns scalars

print("Test Set Evaluation Results:")
print(f"{'':20s} {'Precision':>10s} {'Recall':>10s} {'F1-Score':>10s} {'Support':>10s}")
print("-" * 65)
for i in range(NUM_LABELS):
    label_name = ID2LABEL[i]  # Convert integer ID to readable class name
    print(f"{label_name:20s} {precision[i]:>10.4f} {recall[i]:>10.4f} {f1[i]:>10.4f} {support[i]:>10,}")
print("-" * 65)
print(f"{'Weighted Avg':20s} {weighted_precision:>10.4f} {weighted_recall:>10.4f} {weighted_f1:>10.4f}")
print(f"{'Accuracy':20s} {'':>10s} {'':>10s} {test_accuracy:>10.4f}")
# Expected: accuracy in range 0.80–0.92 depending on dataset and training

In [ ]:
# ---------------------------------------------------------------------------
# 5.3  Print full scikit-learn classification report
# ---------------------------------------------------------------------------
# classification_report gives a formatted string with all per-class and
# aggregate metrics — useful for copy-pasting into the project report

# Map integer labels to string names for a readable report
label_names = [ID2LABEL[i] for i in range(NUM_LABELS)]  # ['Negative', 'Neutral', 'Positive']

report_str = classification_report(
    test_true_labels,    # Ground-truth integer labels
    test_pred_labels,    # Predicted integer labels
    target_names=label_names,  # Use string names instead of integers in output
    digits=4,            # 4 decimal places for metric values
)  # Returns a formatted multi-line string

print("Full Classification Report (RoBERTa — Test Set):")
print(report_str)
# Expected: table showing per-class P/R/F1 + macro/weighted averages

In [ ]:
# ---------------------------------------------------------------------------
# 5.4  Generate and save confusion matrix heatmap
# ---------------------------------------------------------------------------
# The confusion matrix shows which classes are being confused with each other
# Row = true label, Column = predicted label
# Diagonal cells = correct predictions; off-diagonal = mistakes

# Compute the 3×3 confusion matrix
cm = confusion_matrix(
    test_true_labels,  # Ground-truth labels
    test_pred_labels,  # Model predictions
    labels=[0, 1, 2],  # Ensure consistent ordering: Negative=0, Neutral=1, Positive=2
)  # Returns a NumPy array of shape (3, 3) with raw counts

# Normalize to percentages: each row sums to 100%
# This makes cross-class comparison fair regardless of class sizes
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
# cm_normalized[i, j] = (predicted class j | true class i) as a percentage

# Create the figure
fig, axes = plt.subplots(1, 2, figsize=(14, 6))  # Two subplots: raw counts + normalized
fig.suptitle("RoBERTa Sentiment Classification — Confusion Matrix (Test Set)", fontsize=14, fontweight="bold")

# --- Left panel: raw counts ---
sns.heatmap(
    cm,                      # Raw count matrix (3×3 integers)
    annot=True,              # Write count values in each cell
    fmt="d",                 # Format annotations as integers
    cmap="Blues",            # Blue color gradient — darker = higher count
    xticklabels=label_names, # Column labels: predicted class names
    yticklabels=label_names, # Row labels: true class names
    ax=axes[0],              # Draw on left subplot
    linewidths=0.5,          # Thin grid lines between cells
)
axes[0].set_title("Raw Counts")       # Subplot title
axes[0].set_xlabel("Predicted Label")  # X-axis: model's predicted class
axes[0].set_ylabel("True Label")       # Y-axis: actual class

# --- Right panel: normalized percentages ---
sns.heatmap(
    cm_normalized,           # Percentage matrix (3×3 floats, row-normalized)
    annot=True,              # Write percentage values in each cell
    fmt=".1f",               # Format annotations as 1-decimal-place floats
    cmap="Blues",            # Blue color gradient
    xticklabels=label_names, # Column labels: predicted class names
    yticklabels=label_names, # Row labels: true class names
    ax=axes[1],              # Draw on right subplot
    linewidths=0.5,
    vmin=0, vmax=100,        # Force color scale 0–100%
)
axes[1].set_title("Normalized (% of True Class)")  # Subplot title
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Label")

plt.tight_layout()  # Adjust spacing to prevent label overlap

# Save the figure to the PLOTS_DIR on Google Drive
cm_plot_path = os.path.join(PLOTS_DIR, "confusion_matrix_roberta.png")  # PNG save path
plt.savefig(cm_plot_path, dpi=150, bbox_inches="tight")  # High-res PNG
plt.show()  # Display inline in the notebook

print(f"Confusion matrix saved to: {cm_plot_path}")
# Expected: 2-panel figure with blue heatmaps; high diagonal = good performance

In [ ]:
# ---------------------------------------------------------------------------
# 5.5  Save test metrics to JSON for use in Section 6 (comparison)
# ---------------------------------------------------------------------------
# Storing metrics as JSON allows NB04 and NB05 to load them without re-running training
# and enables the comparison table in Section 6 of this notebook

metrics_roberta = {
    "model": "roberta-base",                    # Model identifier
    "num_parameters": total_params,              # ~125M
    "accuracy": float(test_accuracy),            # Scalar float [0, 1]
    "weighted_precision": float(weighted_precision),  # Weighted-avg precision
    "weighted_recall": float(weighted_recall),        # Weighted-avg recall
    "weighted_f1": float(weighted_f1),                # Weighted-avg F1
    "per_class_metrics": {                       # Per-class breakdown
        ID2LABEL[i]: {
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i]),
        }
        for i in range(NUM_LABELS)
    },
    "classification_report": report_str,         # Full report as string
}  # Dict structure matches metrics_distilbert.json from NB02 for easy comparison

# Save path: OUTPUT_DIR/metrics_roberta.json
metrics_path = os.path.join(OUTPUT_DIR, "metrics_roberta.json")

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_roberta, f, indent=2)  # Pretty-print with 2-space indentation

print(f"Metrics saved to: {metrics_path}")
print(f"  Accuracy    : {test_accuracy:.4f}")
print(f"  Weighted F1 : {weighted_f1:.4f}")
# Expected: JSON file created; accuracy and F1 values printed

---
## Section 6 — Model Comparison: RoBERTa vs DistilBERT

In this section, I load the DistilBERT metrics from Notebook 02 and compare both models side by side. This comparison is the core deliverable of Notebook 03 — understanding when the extra compute cost of RoBERTa is justified.

**My hypothesis**: RoBERTa will outperform DistilBERT on the minority classes (Negative, Neutral) thanks to its deeper architecture and better pretraining, but the gap on overall accuracy may be smaller than expected given that DistilBERT retains ~97% of BERT's representational capacity.

In [ ]:
# ---------------------------------------------------------------------------
# 6.1  Load DistilBERT metrics from Notebook 02
# ---------------------------------------------------------------------------
# NB02 saved its test metrics to OUTPUT_DIR/metrics_distilbert.json
# If the file doesn't exist, we create a placeholder with realistic estimates
# so the comparison section still runs and shows the expected structure

distilbert_metrics_path = os.path.join(OUTPUT_DIR, "metrics_distilbert.json")

if os.path.exists(distilbert_metrics_path):
    # Load the actual metrics produced by NB02
    with open(distilbert_metrics_path, "r", encoding="utf-8") as f:
        metrics_distilbert = json.load(f)  # Dict with accuracy, f1, per_class_metrics, etc.
    print(f"DistilBERT metrics loaded from: {distilbert_metrics_path}")
else:
    # Placeholder: use typical DistilBERT performance estimates if NB02 hasn't run yet
    # These values should be replaced by actual NB02 output
    print("WARNING: metrics_distilbert.json not found. Using placeholder estimates.")
    print("Run Notebook 02 first to generate actual DistilBERT metrics.")
    metrics_distilbert = {
        "model": "distilbert-base-uncased",
        "num_parameters": 66_000_000,
        "accuracy": 0.85,          # Placeholder — replace with actual NB02 result
        "weighted_precision": 0.84,
        "weighted_recall": 0.85,
        "weighted_f1": 0.84,
        "per_class_metrics": {
            "Negative": {"precision": 0.80, "recall": 0.75, "f1": 0.77, "support": 0},
            "Neutral":  {"precision": 0.55, "recall": 0.50, "f1": 0.52, "support": 0},
            "Positive": {"precision": 0.90, "recall": 0.95, "f1": 0.92, "support": 0},
        },
    }

print(f"  DistilBERT Accuracy: {metrics_distilbert['accuracy']:.4f}")
print(f"  DistilBERT F1      : {metrics_distilbert['weighted_f1']:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# 6.2  Build side-by-side comparison table
# ---------------------------------------------------------------------------
# Create a Pandas DataFrame to display RoBERTa vs DistilBERT metrics cleanly

# Extract top-level metrics for both models
comparison_data = {
    "Metric": ["Accuracy", "Weighted Precision", "Weighted Recall", "Weighted F1"],
    "DistilBERT (~66M params)": [
        metrics_distilbert["accuracy"],
        metrics_distilbert["weighted_precision"],
        metrics_distilbert["weighted_recall"],
        metrics_distilbert["weighted_f1"],
    ],
    "RoBERTa (~125M params)": [
        test_accuracy,
        float(weighted_precision),
        float(weighted_recall),
        float(weighted_f1),
    ],
}

# Convert to DataFrame for clean table display
comparison_df = pd.DataFrame(comparison_data)  # Shape: (4, 3)

# Add a Delta column showing RoBERTa improvement over DistilBERT
comparison_df["Delta (RoBERTa - DistilBERT)"] = (
    comparison_df["RoBERTa (~125M params)"] - comparison_df["DistilBERT (~66M params)"]
).round(4)  # Positive delta = RoBERTa is better; negative = DistilBERT is better

# Set Metric as the index for cleaner display
comparison_df = comparison_df.set_index("Metric")

# Round all numeric values to 4 decimal places for readability
comparison_df = comparison_df.round(4)

print("Model Comparison — RoBERTa vs DistilBERT (Test Set):")
print(comparison_df.to_string())  # Print full table without truncation
# Expected: table with 4 rows; positive deltas indicate RoBERTa improvement

In [ ]:
# ---------------------------------------------------------------------------
# 6.3  Per-class comparison table
# ---------------------------------------------------------------------------
# Aggregate metrics can hide per-class differences — the most interesting
# comparison is on Neutral (hardest class) and Negative (minority class)

rows = []  # Collect one row per class-metric combination

for class_name in label_names:  # ['Negative', 'Neutral', 'Positive']
    db_class = metrics_distilbert["per_class_metrics"][class_name]  # DistilBERT metrics for class
    rb_class = metrics_roberta["per_class_metrics"][class_name]      # RoBERTa metrics for class

    for metric_name in ["precision", "recall", "f1"]:
        rows.append({
            "Class": class_name,
            "Metric": metric_name.capitalize(),
            "DistilBERT": round(db_class[metric_name], 4),
            "RoBERTa": round(rb_class[metric_name], 4),
            "Delta": round(rb_class[metric_name] - db_class[metric_name], 4),
        })

per_class_df = pd.DataFrame(rows)  # Shape: (9 rows × 5 cols)
per_class_df = per_class_df.set_index(["Class", "Metric"])  # MultiIndex for clarity

print("Per-Class Comparison:")
print(per_class_df.to_string())
# Expected: 9-row table; largest RoBERTa gains expected on Neutral and Negative F1

In [ ]:
# ---------------------------------------------------------------------------
# 6.4  Bar chart comparison plot
# ---------------------------------------------------------------------------
# Visual comparison makes it easy to spot performance differences at a glance

# Prepare data for grouped bar chart
metric_names = ["Accuracy", "Weighted F1", "Weighted Precision", "Weighted Recall"]

# Extract values for both models in the same order as metric_names
distilbert_scores = [
    metrics_distilbert["accuracy"],
    metrics_distilbert["weighted_f1"],
    metrics_distilbert["weighted_precision"],
    metrics_distilbert["weighted_recall"],
]
roberta_scores = [
    test_accuracy,
    float(weighted_f1),
    float(weighted_precision),
    float(weighted_recall),
]

x = np.arange(len(metric_names))  # [0, 1, 2, 3] — x positions for bar groups
bar_width = 0.35                   # Width of each bar in the grouped pair

fig, ax = plt.subplots(figsize=(10, 6))  # Single panel figure

# Draw DistilBERT bars (offset left by half bar_width)
bars1 = ax.bar(
    x - bar_width / 2,        # Left-shifted positions
    distilbert_scores,        # DistilBERT metric values
    bar_width,                # Bar width
    label="DistilBERT (~66M)", # Legend label
    color="steelblue",        # Blue for DistilBERT
    alpha=0.85,               # Slight transparency
)

# Draw RoBERTa bars (offset right by half bar_width)
bars2 = ax.bar(
    x + bar_width / 2,        # Right-shifted positions
    roberta_scores,           # RoBERTa metric values
    bar_width,
    label="RoBERTa (~125M)",  # Legend label
    color="tomato",           # Red for RoBERTa
    alpha=0.85,
)

# Annotate each bar with its exact value
for bar in bars1:
    ax.text(
        bar.get_x() + bar.get_width() / 2,  # Center of bar
        bar.get_height() - 0.03,             # Just below the top
        f"{bar.get_height():.3f}",           # Format to 3 decimal places
        ha="center", va="top",
        fontsize=9, color="white", fontweight="bold",
    )
for bar in bars2:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() - 0.03,
        f"{bar.get_height():.3f}",
        ha="center", va="top",
        fontsize=9, color="white", fontweight="bold",
    )

# Chart formatting
ax.set_xlabel("Metric", fontsize=12)         # X-axis label
ax.set_ylabel("Score", fontsize=12)          # Y-axis label
ax.set_title("RoBERTa vs DistilBERT — Test Set Performance Comparison", fontsize=14, fontweight="bold")
ax.set_xticks(x)                             # Place x-ticks at group centers
ax.set_xticklabels(metric_names, fontsize=11) # Metric name labels
ax.set_ylim(0.5, 1.0)                        # Y-axis starts at 0.5 to zoom in on differences
ax.legend(fontsize=11)                       # Show model names in legend
ax.yaxis.grid(True, alpha=0.4)               # Horizontal grid lines for readability
ax.set_axisbelow(True)                       # Grid behind bars

plt.tight_layout()  # Prevent label clipping

# Save the comparison chart
comparison_plot_path = os.path.join(PLOTS_DIR, "comparison_roberta_vs_distilbert.png")
plt.savefig(comparison_plot_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Comparison bar chart saved to: {comparison_plot_path}")
# Expected: grouped bar chart with 4 metric pairs; RoBERTa bars typically taller

### Analysis: Which Model Wins and Why?

Looking at the comparison results, I can draw the following conclusions:

**RoBERTa advantages over DistilBERT:**
- **Deeper architecture** (12 layers vs 6): allows RoBERTa to capture longer-range dependencies and subtler linguistic cues in review text — particularly useful for distinguishing Neutral reviews (which can be either mildly positive or mildly negative in wording).
- **Larger vocabulary and more pretraining data**: BPE with 50,265 tokens handles rare product names, brand terms, and technical jargon better than WordPiece's 30,522 tokens. Pretraining on 160GB+ of text gives RoBERTa broader world knowledge.
- **No NSP pretraining**: BERT's Next Sentence Prediction task was removed in RoBERTa after research showed it did not improve downstream performance. RoBERTa's pretraining budget went entirely toward Masked Language Modeling, producing better contextualized embeddings.
- **Expected gains**: The largest improvement should appear on the **Neutral** class — the most ambiguous, borderline reviews benefit most from deeper linguistic reasoning.

**DistilBERT's practical advantages:**
- **~2× faster** at inference — critical for a production system processing thousands of reviews per hour.
- **Smaller memory footprint** (~250MB vs ~500MB) — easier to deploy on constrained infrastructure.
- **Shorter training time** — enables faster iteration and experimentation.

**Trade-off decision**: For this Ironhack project, I use RoBERTa as the primary production model if its F1 score exceeds DistilBERT's by more than 1 percentage point — otherwise, DistilBERT is the pragmatic choice. Both models are saved and their predictions stored for downstream use in the clustering and summarization notebooks.

---
## Section 7 — Inference & Save Predictions

In this section, I run inference on the full test set, save all predictions to a CSV file (which will feed Notebooks 04 and 05), and save the fine-tuned model and tokenizer to Google Drive for future use without retraining.

In [ ]:
# ---------------------------------------------------------------------------
# 7.1  Retrieve original test texts for the predictions CSV
# ---------------------------------------------------------------------------
# The tokenized_dataset doesn't contain raw text (we removed it in Section 2)
# so I access the original dataset['test'] split here for the text column

# Retrieve raw text from the original (untokenized) test split
test_texts = dataset["test"]["text"]  # List of raw review strings — one per test sample

print(f"Number of test texts retrieved: {len(test_texts):,}")
print(f"Number of predictions         : {len(test_pred_labels):,}")
# Expected: both counts match the test split size

In [ ]:
# ---------------------------------------------------------------------------
# 7.2  Build predictions DataFrame and save as CSV
# ---------------------------------------------------------------------------
# The predictions CSV is consumed by NB04 (clustering) to stratify reviews
# and by NB05 (summarization) to select negative reviews per cluster

predictions_df = pd.DataFrame({
    "text": test_texts,                                              # Raw review text
    "true_label": [ID2LABEL[l] for l in test_true_labels.tolist()],  # String true label
    "predicted_label": [ID2LABEL[l] for l in test_pred_labels.tolist()],  # String predicted label
    "true_label_id": test_true_labels.tolist(),                       # Integer true label ID
    "predicted_label_id": test_pred_labels.tolist(),                  # Integer predicted label ID
    "confidence": test_confidence.tolist(),                           # Max softmax probability
    "is_correct": (test_true_labels == test_pred_labels).tolist(),    # Boolean: prediction correct?
})  # DataFrame shape: (n_test, 7)

# Save path: OUTPUT_DIR/predictions_roberta.csv
predictions_csv_path = os.path.join(OUTPUT_DIR, "predictions_roberta.csv")

predictions_df.to_csv(
    predictions_csv_path,  # Save destination on Google Drive
    index=False,           # Don't write row index — cleaner CSV
    encoding="utf-8",      # UTF-8 handles all Unicode characters in review text
)  # Expected: CSV file with 7 columns saved to OUTPUT_DIR

print(f"Predictions saved to: {predictions_csv_path}")
print(f"DataFrame shape: {predictions_df.shape}")
print(f"\nFirst 5 rows:")
print(predictions_df[["true_label", "predicted_label", "confidence", "is_correct"]].head())
# Expected: 5 rows showing predicted vs true labels and confidence scores

In [ ]:
# ---------------------------------------------------------------------------
# 7.3  Save fine-tuned model and tokenizer to Google Drive
# ---------------------------------------------------------------------------
# Saving the model to Drive means I can load it directly next time
# without rerunning the entire training pipeline (which takes ~30-60 min)
# The saved model can also be loaded for the web app deployment in Phase 6

print(f"Saving fine-tuned RoBERTa model to: {ROBERTA_MODEL_DIR}")

# Save the model weights, configuration, and classification head
model.save_pretrained(ROBERTA_MODEL_DIR)  # Saves pytorch_model.bin + config.json

# Save the tokenizer vocabulary and configuration
tokenizer.save_pretrained(ROBERTA_MODEL_DIR)  # Saves vocab.json, merges.txt, tokenizer_config.json

# List the saved files for verification
saved_files = os.listdir(ROBERTA_MODEL_DIR)  # All files in the model directory
print(f"Files saved to {ROBERTA_MODEL_DIR}:")
for fname in sorted(saved_files):  # Sort for consistent display order
    fsize = os.path.getsize(os.path.join(ROBERTA_MODEL_DIR, fname))  # File size in bytes
    print(f"  {fname:50s}  {fsize / 1e6:.1f} MB")  # Display size in megabytes
# Expected: pytorch_model.bin (~475 MB), config.json, vocab.json, merges.txt, etc.

In [ ]:
# ---------------------------------------------------------------------------
# 7.4  Demonstrate loading the saved model from disk
# ---------------------------------------------------------------------------
# Verifying that the model can be reloaded confirms the save was successful
# This is the same pattern that would be used in the web app or NB05

print("Verifying saved model can be reloaded from disk...")

# Reload tokenizer from the saved directory
loaded_tokenizer = RobertaTokenizerFast.from_pretrained(ROBERTA_MODEL_DIR)

# Reload model from the saved directory
loaded_model = RobertaForSequenceClassification.from_pretrained(ROBERTA_MODEL_DIR)
loaded_model = loaded_model.to(DEVICE)  # Move to GPU for inference
loaded_model.eval()                     # Switch to evaluation mode (disables dropout)

# Quick smoke test: run inference on a single example
test_sentence = "This product exceeded all my expectations. Absolutely fantastic quality!"
test_inputs = loaded_tokenizer(
    test_sentence,
    return_tensors="pt",   # Return PyTorch tensors (model expects tensors, not lists)
    truncation=True,
    max_length=MAX_LENGTH,
    padding="max_length",
)
test_inputs = {k: v.to(DEVICE) for k, v in test_inputs.items()}  # Move inputs to GPU

# Forward pass without gradient tracking (saves memory, speeds up inference)
with torch.no_grad():
    test_logits_out = loaded_model(**test_inputs).logits  # Shape: (1, 3)

# Convert to probability and pick the highest-confidence class
test_probs_out = F.softmax(test_logits_out, dim=-1).squeeze()  # Shape: (3,)
pred_class_id = test_probs_out.argmax().item()                  # Integer class ID
pred_class_name = ID2LABEL[pred_class_id]                       # String label

print(f"\nSmoke test sentence: '{test_sentence}'")
print(f"Predicted sentiment : {pred_class_name}  (confidence: {test_probs_out[pred_class_id]:.3f})")
print(f"All class probabilities: Negative={test_probs_out[0]:.3f}, Neutral={test_probs_out[1]:.3f}, Positive={test_probs_out[2]:.3f}")
# Expected: Predicted sentiment = Positive with high confidence (~0.95+)

---
## Section 8 — Results Summary

In this final section, I consolidate all results from this notebook: final metrics, sample correct and incorrect predictions, training/validation loss curves, and a complete file checklist confirming all artifacts have been saved to Google Drive.

In [ ]:
# ---------------------------------------------------------------------------
# 8.1  Print final metrics summary
# ---------------------------------------------------------------------------

print("=" * 60)
print("  ROBERTA SENTIMENT CLASSIFICATION — FINAL RESULTS")
print("=" * 60)
print(f"  Model         : roberta-base (FacebookAI)")
print(f"  Parameters    : ~{total_params / 1e6:.0f}M")
print(f"  Max length    : {MAX_LENGTH} tokens")
print(f"  Learning rate : {LEARNING_RATE}")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Epochs        : {NUM_EPOCHS}")
print("-" * 60)
print(f"  Test Accuracy          : {test_accuracy:.4f}  ({test_accuracy * 100:.2f}%)")
print(f"  Weighted Precision     : {weighted_precision:.4f}")
print(f"  Weighted Recall        : {weighted_recall:.4f}")
print(f"  Weighted F1-Score      : {weighted_f1:.4f}")
print("-" * 60)
print("  Per-Class F1 Scores:")
for i in range(NUM_LABELS):
    print(f"    {ID2LABEL[i]:12s}: {f1[i]:.4f}")
print("=" * 60)

In [ ]:
# ---------------------------------------------------------------------------
# 8.2  Show 5 correct + 5 incorrect predictions with confidence scores
# ---------------------------------------------------------------------------
# Reviewing specific examples reveals where the model succeeds and fails
# Correct predictions: model is confident and right
# Incorrect predictions: model is wrong — inspect whether it was also uncertain

# Filter correct predictions
correct_df = predictions_df[predictions_df["is_correct"] == True].head(5)  # First 5 correct

# Filter incorrect predictions
incorrect_df = predictions_df[predictions_df["is_correct"] == False].head(5)  # First 5 wrong

print("CORRECT PREDICTIONS (first 5):")
print("-" * 80)
for _, row in correct_df.iterrows():
    text_preview = row["text"][:100].replace("\n", " ")  # Truncate for display
    print(f"  True: {row['true_label']:10s} | Pred: {row['predicted_label']:10s} | Conf: {row['confidence']:.3f}")
    print(f"  Text: {text_preview}...")
    print()

print("INCORRECT PREDICTIONS (first 5):")
print("-" * 80)
for _, row in incorrect_df.iterrows():
    text_preview = row["text"][:100].replace("\n", " ")  # Truncate for display
    print(f"  True: {row['true_label']:10s} | Pred: {row['predicted_label']:10s} | Conf: {row['confidence']:.3f}")
    print(f"  Text: {text_preview}...")
    print()
# Expected: correct predictions tend to have higher confidence; errors often on Neutral class

In [ ]:
# ---------------------------------------------------------------------------
# 8.3  Plot training and validation loss curves
# ---------------------------------------------------------------------------
# The loss curves show whether the model converged, overfit, or underfit
# A healthy training run shows both curves decreasing and converging
# Diverging validation loss (while training loss keeps falling) = overfitting

# Extract training history from the Trainer's log history
# trainer.state.log_history is a list of dicts with training/eval metrics per step
log_history = trainer.state.log_history  # List of dicts — one entry per logging step

# Separate training loss entries (have 'loss' key but not 'eval_loss')
train_loss_entries = [
    entry for entry in log_history
    if "loss" in entry and "eval_loss" not in entry
]  # Training loss logged every logging_steps steps

# Separate evaluation loss entries (have 'eval_loss' key)
eval_loss_entries = [
    entry for entry in log_history
    if "eval_loss" in entry
]  # Evaluation loss logged once per epoch

# Extract step numbers and loss values for plotting
train_steps = [e["step"] for e in train_loss_entries]   # X-axis: training steps
train_losses = [e["loss"] for e in train_loss_entries]  # Y-axis: training loss values

eval_steps = [e["step"] for e in eval_loss_entries]        # X-axis: eval steps (end of each epoch)
eval_losses = [e["eval_loss"] for e in eval_loss_entries]  # Y-axis: validation loss values

# Also extract accuracy per epoch if available
eval_accuracies = [e.get("eval_accuracy", None) for e in eval_loss_entries]

# Create a 2-panel figure: loss on left, accuracy on right
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("RoBERTa Fine-Tuning — Training History", fontsize=14, fontweight="bold")

# --- Left panel: Training and Validation Loss ---
axes[0].plot(
    train_steps, train_losses,  # Training loss (dense — logged every 50 steps)
    label="Training Loss",
    color="steelblue",
    linewidth=1.5,
    alpha=0.8,
)
axes[0].plot(
    eval_steps, eval_losses,    # Validation loss (sparse — logged per epoch)
    label="Validation Loss",
    color="tomato",
    linewidth=2.5,
    marker="o",                 # Circle markers at epoch boundaries
    markersize=8,
)
axes[0].set_xlabel("Training Step", fontsize=11)
axes[0].set_ylabel("Loss", fontsize=11)
axes[0].set_title("Loss Curve")
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.4)

# --- Right panel: Validation Accuracy per Epoch ---
if any(a is not None for a in eval_accuracies):  # Only plot if accuracy data exists
    valid_epochs = [i + 1 for i, a in enumerate(eval_accuracies) if a is not None]
    valid_accs = [a for a in eval_accuracies if a is not None]
    axes[1].plot(
        valid_epochs, valid_accs,
        color="seagreen",
        linewidth=2.5,
        marker="o",
        markersize=8,
        label="Validation Accuracy",
    )
    axes[1].set_xlabel("Epoch", fontsize=11)
    axes[1].set_ylabel("Accuracy", fontsize=11)
    axes[1].set_title("Validation Accuracy per Epoch")
    axes[1].set_ylim(0.5, 1.0)  # Zoom into the interesting range
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.4)
    axes[1].set_xticks(valid_epochs)  # One tick per epoch
else:
    axes[1].text(0.5, 0.5, "No accuracy data available", ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()

# Save the loss curve plot
loss_curve_path = os.path.join(PLOTS_DIR, "training_history_roberta.png")
plt.savefig(loss_curve_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Training history plot saved to: {loss_curve_path}")
# Expected: decreasing train and val loss; accuracy increasing per epoch

In [ ]:
# ---------------------------------------------------------------------------
# 8.4  Final file checklist
# ---------------------------------------------------------------------------
# Verify that all expected output files have been saved to Google Drive
# This checklist confirms the notebook produced all required artifacts

print("Final Artifact Checklist:")
print("=" * 65)

# Define all expected output files with their descriptions
expected_files = [
    (os.path.join(OUTPUT_DIR, "metrics_roberta.json"),                   "RoBERTa test metrics (JSON)"),
    (os.path.join(OUTPUT_DIR, "predictions_roberta.csv"),                 "Test predictions CSV"),
    (os.path.join(PLOTS_DIR,  "confusion_matrix_roberta.png"),            "Confusion matrix heatmap"),
    (os.path.join(PLOTS_DIR,  "comparison_roberta_vs_distilbert.png"),    "Model comparison bar chart"),
    (os.path.join(PLOTS_DIR,  "training_history_roberta.png"),            "Training/validation loss curve"),
    (os.path.join(ROBERTA_MODEL_DIR, "config.json"),                      "Fine-tuned model config"),
    (os.path.join(ROBERTA_MODEL_DIR, "tokenizer_config.json"),            "Tokenizer config"),
]

# Check each file and print status
all_present = True  # Track whether all files exist
for filepath, description in expected_files:
    exists = os.path.exists(filepath)  # True if file was saved successfully
    status = "✓ PRESENT" if exists else "✗ MISSING"  # Visual status indicator
    size_str = ""
    if exists:
        size_bytes = os.path.getsize(filepath)  # File size in bytes
        size_str = f" ({size_bytes / 1024:.1f} KB)" if size_bytes < 1e6 else f" ({size_bytes / 1e6:.1f} MB)"
    print(f"  [{status}] {description}{size_str}")
    if not exists:
        all_present = False  # At least one file is missing

print("=" * 65)
if all_present:
    print("  ALL ARTIFACTS SAVED SUCCESSFULLY. Notebook 03 complete!")
else:
    print("  WARNING: Some artifacts are missing — review cells above for errors.")
print("=" * 65)

In [ ]:
# ---------------------------------------------------------------------------
# 8.5  Next steps — downstream notebooks
# ---------------------------------------------------------------------------
# Print a reminder of what artifacts this notebook produced and where they go

print("""
Notebook 03 — RoBERTa Sentiment Classification — COMPLETE

Artifacts produced:
  OUTPUT_DIR/metrics_roberta.json        → Used in NB03 Section 6 comparison
  OUTPUT_DIR/predictions_roberta.csv     → Used by NB04 (clustering) + NB05 (summarization)
  OUTPUT_DIR/models/roberta_sentiment/   → Fine-tuned model weights for deployment
  OUTPUT_DIR/plots/*.png                 → Confusion matrix, comparison chart, loss curves

Next step:
  → Run Notebook 04: Product Category Clustering
    Input: predictions_roberta.csv (or predictions_distilbert.csv)
    Model: all-MiniLM-L6-v2 (sentence embeddings) + MiniBatchKMeans
    Output: clusters.csv with cluster assignments
"""
)